# NYC Hydrant Density Analysis (GeoPandas)Portfolio Project 2, Modern GIS Accelerator, Part 2This notebook runs the same five-step analysis as `analysis.sql`, but in GeoPandas. The point of doing it twice is to internalize the parallel: SQL transforms at scale, Python orchestrates and visualizes.**Expected output:**- A static choropleth of hydrant density per km² across NYC neighborhoods- An interactive `.explore()` map for the README screenshot- A GeoParquet export at `data/processed/hydrant_density.parquet`

## SetupMake sure your virtual environment has the packages below. If not:```bashpip install geopandas matplotlib jupyter pyarrow folium mapclassify```

In [ ]:
import geopandas as gpdimport matplotlib.pyplot as pltimport os# Make sure the output directory existsos.makedirs("data/processed", exist_ok=True)os.makedirs("images", exist_ok=True)print("GeoPandas version:", gpd.__version__)

## Load the dataTwo source datasets, both in EPSG:4326. Adjust the paths if your files live somewhere else.

In [ ]:
# [TODO] Replace these paths with the actual location of your data files.neighborhoods = gpd.read_file("data/raw/nyc_neighborhoods.geojson")hydrants = gpd.read_file("data/raw/nyc_hydrants.geojson")print("Neighborhoods:", neighborhoods.shape, "| CRS:", neighborhoods.crs)print("Hydrants:     ", hydrants.shape,     "| CRS:", hydrants.crs)

## Query 1 (parallel): FilterSanity check that the data loaded. Pull all neighborhoods in Manhattan.

In [ ]:
manhattan = neighborhoods[neighborhoods["borough"] == "Manhattan"]manhattan[["neighborhood", "borough"]].head(10)

## Query 2 (parallel): Spatial joinMatch each hydrant to the neighborhood that contains it. Equivalent of `JOIN ... ON ST_Contains(n.geom, h.geom)`.

In [ ]:
# [TODO] Use gpd.sjoin to join hydrants and neighborhoods.# Predicate: "within" (hydrant within neighborhood)# Suggested: how="left" so unmatched hydrants are preserved (you can count them later)joined = gpd.sjoin(hydrants, neighborhoods, predicate="within", how="left")print(f"Joined rows: {len(joined):,}")joined.head()

## Query 3 (parallel): AggregateCount hydrants per neighborhood. Equivalent of `GROUP BY neighborhood ... COUNT(*)`.

In [ ]:
# [TODO] Group joined by 'neighborhood' and count rows.counts = (    joined.groupby("neighborhood").size().rename("hydrant_count").to_frame())counts.sort_values("hydrant_count", ascending=False).head(10)

## Query 4 (parallel): Normalize (the headline)Reproject to EPSG:2263 (NY State Plane Long Island, feet) to compute area in real units. Convert square feet to square kilometers, then density per km².

In [ ]:
# [TODO] Merge counts back onto neighborhoods, then compute density.SQ_FT_PER_KM2 = 10_763_910.42result = neighborhoods.merge(counts, on="neighborhood", how="left").fillna({"hydrant_count": 0})result["area_km2"] = result.to_crs(2263).area / SQ_FT_PER_KM2result["density_per_km2"] = result["hydrant_count"] / result["area_km2"]result.sort_values("density_per_km2", ascending=False)[    ["neighborhood", "borough", "hydrant_count", "area_km2", "density_per_km2"]].head(10)

## Query 5 (parallel): Coverage analysisWhat percent of each neighborhood is within 100 meters of a hydrant?This is the deeper finding. The headline (Query 4) tells you where coverage is densest. This tells you how complete coverage is across each neighborhood, regardless of count.

In [ ]:
# [TODO] Build this in two steps:# 1) Buffer every hydrant by 100 meters (after reprojecting to 2263)# 2) Compute the union of all those buffers# 3) Intersect each neighborhood with the union and divide by neighborhood area.hydrants_2263 = hydrants.to_crs(2263)hydrant_coverage = hydrants_2263.buffer(100).union_all()result_2263 = result.to_crs(2263).copy()result_2263["covered_area"] = result_2263.geometry.intersection(hydrant_coverage).arearesult_2263["covered_pct"] = result_2263["covered_area"] / result_2263.geometry.arearesult_2263[["neighborhood", "borough", "covered_pct"]].sort_values(    "covered_pct", ascending=False).head(10)

## Visualize: static choroplethThis is the figure that goes in your README. Save it to `images/density_choropleth.png`.Voice rule: get the data right yourself, then ask AI to help with the styling. See R1.3 Prompt 19 for matplotlib styling patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))result.to_crs(3857).plot(    column="density_per_km2",    cmap="viridis",    legend=True,    edgecolor="white",    linewidth=0.3,    ax=ax,    legend_kwds={"label": "Hydrants per km²", "shrink": 0.6},)ax.set_title("NYC Fire Hydrant Density by Neighborhood", fontsize=16, pad=12)ax.set_axis_off()plt.tight_layout()plt.savefig("images/density_choropleth.png", dpi=150, bbox_inches="tight")plt.show()

## Visualize: interactive mapFor a richer README, take a screenshot of this map after running.

In [ ]:
result[["neighborhood", "borough", "hydrant_count", "density_per_km2", "geometry"]].explore(    column="density_per_km2",    cmap="viridis",    tooltip=["neighborhood", "borough", "hydrant_count", "density_per_km2"],    legend=True,    style_kwds={"weight": 0.5, "color": "white", "fillOpacity": 0.7},)

## Export to GeoParquetThis is the artifact PP3 (web map) will reuse. Land it cleanly so the next project can read it without conversion.

In [ ]:
# Keep only the columns that matter downstream.out = result[[    "neighborhood", "borough", "hydrant_count", "area_km2", "density_per_km2", "geometry"]].copy()out.to_parquet("data/processed/hydrant_density.parquet")print("Wrote data/processed/hydrant_density.parquet")print("Shape:", out.shape)print("CRS:  ", out.crs)

## What I learned**[TODO]** Two or three sentences. Be specific. What was harder than expected? Where did the SQL and Python answers disagree, and why? What would you do differently next time?This is the part hiring managers actually read. Do not skip it.---*Companion to:* `analysis.sql` (same analysis in PostGIS), and the case-study README.